In [1]:
from typing import TypedDict, List
from langgraph.graph import StateGraph, END
from langchain_core.messages import HumanMessage, AIMessage, BaseMessage

class ChatState(TypedDict):
    messages : List[BaseMessage]
    user_input: str

In [2]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv('GROQ_API_KEY_LANGCHAIN_PRACTICE')

llm = init_chat_model (
    model='llama-3.3-70b-versatile', 
    model_provider='groq',
    temperature=0.7,    
    api_key=api_key
)

In [3]:
# adding user input to memory
def add_user_memory(state: ChatState) -> ChatState:
    return {
        **state,
        "messages": state["messages"] + [
            HumanMessage(content=state["user_input"])
        ]
    }

In [4]:
# llm response node
def chatbot_node(state: ChatState) -> ChatState:
    response = llm.invoke(state["messages"])
    return {
        **state,
        "messages": state["messages"] + [
            AIMessage(content=response.content)
        ]
    }

In [5]:
# trim memory if it becomes big
def trim_memory(state : ChatState) -> ChatState:
    messages = state['messages']
    if len(messages) > 6: 
        messages = messages[-6:]
    return {
        **state,
        'messages' : messages
    }

In [6]:
# router
def router(state : ChatState) -> str:
    text = state['user_input'].lower()
    if 'remember' in text:
        return "chat"
    elif 'clear' in text:
        return 'clear'
    return "chat"

In [7]:
# clear memory node
def clear_memory(state : ChatState) -> ChatState:
    return {
        'messages' : [],
        'user_input' : state['user_input']
    }

In [8]:
# adding nodes of graph

graph = StateGraph(ChatState)
graph.add_node('add_user', add_user_memory)
graph.add_node('chat', chatbot_node)
graph.add_node('trim', trim_memory)
graph.add_node('clear', clear_memory)

In [9]:
# entry point and edge defining

graph.set_entry_point('add_user')
graph.add_edge("add_user", "trim")

graph.add_conditional_edges(
    "trim",
    router,
    {
        "chat": "chat",
        "clear": "clear"
    }
)
graph.add_edge("chat", END)
graph.add_edge("clear", END)

In [10]:
# compile
app = graph.compile()

In [11]:
# chat loop
state = {
    'messages': [],
    'user_input': ""
}

In [12]:
print("Stateful Chatbot (type 'exit' to quit, 'clear' to reset)\n")

while True:
    user_input = input("You: ")

    if user_input.lower() == "exit":
        break

    state["user_input"] = user_input

    result = app.invoke(state)

    state["messages"] = result["messages"]

    if state["messages"]:
        print("Bot:", state["messages"][-1].content)
    else:
        print("Bot: Memory cleared.")

Stateful Chatbot (type 'exit' to quit, 'clear' to reset)

Bot: Hello. How can I assist you today?
Bot: I don't know your name. I'm a large language model, I don't have the ability to retain information about individual users, so I don't have any knowledge about your name or personal details. If you'd like to share your name with me, I'd be happy to chat with you and address you by name.
Bot: Nice to meet you, Joe. It's great to have a name to go with our conversation. How's your day going so far, Joe? Is there something I can help you with or would you like to just chat?
Bot: My memory is a bit limited, Joe. I'm a large language model, I don't have a traditional memory like a human does. I can process and respond to text-based input in real-time, but I don't have the ability to store information about our conversation or recall it later.

Each time you interact with me, it's a new conversation and I don't retain any context or information from previous conversations. So, while I know y

In [ ]:
from langchain_community.chat_message_histories import ChatMessageHistory
 